In [ ]:
import ast

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tqdm
from astropy.io import fits
from scipy.spatial import cKDTree


# Load in catalog data

In [ ]:
with fits.open('all_maps/data/masks/erass1cl_primary_v3.2.fits') as hdul:
    erass = hdul[1].data
with fits.open('all_maps/data/cluster_catalogs/MCXC-II.dat.gz.fits') as hdul:
    mcxc = hdul[1].data
with fits.open('all_maps/data/cluster_catalogs/SPT-ECS.fits') as hdul:
    spt_ecs = hdul[1].data
with fits.open('all_maps/data/cluster_catalogs/ACT_DR6_cluster-catalog_v1.0.fits') as hdul:
    actdr6 = hdul[1].data
with fits.open('all_maps/data/cluster_catalogs/HFI_PCCS_SZ-union_R2.08.fits') as hdul:
    planck = hdul[1].data
f_des = h5py.File('all_maps/data/cluster_catalogs/y3_redmapper_v6.4.22+2_release.h5', 'r')
camira_deep = pd.read_csv('all_maps/data/cluster_catalogs/camira_s20a_dud.tbl', skiprows=9, delim_whitespace=True, 
            names=['ID', 'name', 'RA', 'DEC', 'Z', 'RICHNESS', 'Z_BCG'])
camira_wide = pd.read_csv('all_maps/data/cluster_catalogs/camira_s20a_wide.tbl', skiprows=9, delim_whitespace=True, 
            names=['ID', 'name', 'RA', 'DEC', 'Z', 'RICHNESS', 'Z_BCG'])

In [ ]:
erass_df = pd.DataFrame({
    'RA': erass['RA'],
    'DEC': erass['DEC'],
    'redshift': erass['BEST_Z'],
    'redshift_type': erass['BEST_Z_TYPE'],
    'redshift_unc': erass['BEST_ZERR'],
    'bcg_specz': -1,
    'richness': -1,
    'richness_unc': -1,
    'M200m': -1,
    'M500c': erass['M500'],
    'catalog_source': 'eRASS 1',
    'catalog_id': erass['NAME'],
    })

In [ ]:
spt_clusters_df = pd.DataFrame({
    'RA': spt_ecs['RA'].byteswap().view(spt_ecs['RA'].dtype.newbyteorder()),
    'DEC': spt_ecs['DEC'].byteswap().view(spt_ecs['DEC'].dtype.newbyteorder()),
    'redshift': spt_ecs['REDSHIFT'].byteswap().view(spt_ecs['REDSHIFT'].dtype.newbyteorder()),
    'redshift_type': -1,
    'redshift_unc': spt_ecs['REDSHIFT_UNC'].byteswap().view(spt_ecs['REDSHIFT_UNC'].dtype.newbyteorder()),
    'bcg_specz': spt_ecs['SPECZ'].byteswap().view(spt_ecs['SPECZ'].dtype.newbyteorder()),
    'richness': spt_ecs['LAMBDA_CHISQ'].byteswap().view(spt_ecs['LAMBDA_CHISQ'].dtype.newbyteorder()),
    'richness_unc': spt_ecs['LAMBDA_CHISQ_E'].byteswap().view(spt_ecs['LAMBDA_CHISQ_E'].dtype.newbyteorder()),
    'M200m': -1,
    'M500c': spt_ecs['M500'].byteswap().view(spt_ecs['M500'].dtype.newbyteorder()) * 1.43e14,
    'catalog_source': 'SPT ECS',
    'catalog_id': spt_ecs['SPT_ID'].byteswap().view(spt_ecs['SPT_ID'].dtype.newbyteorder()),
    })

In [ ]:
mcxc_clusters_df = pd.DataFrame({
    'RA': mcxc['RAdeg'],
    'DEC': mcxc['DEdeg'],
    'redshift': mcxc['z'],
    'redshift_type': mcxc['zType'],
    'redshift_unc': -1,
    'bcg_specz': -1,
    'richness': -1,
    'richness_unc': -1,
    'M200m': -1,
    'M500c': mcxc['M500'] * 1e14,
    'catalog_source': 'MCXC-II',
    'catalog_id': mcxc['Index'],
    })

In [ ]:
act_clusters_df = pd.DataFrame({
    'RA': actdr6['RADeg'],
    'DEC': actdr6['decDeg'],
    'redshift': actdr6['redshift'],
    'redshift_type': actdr6['redshiftType'],
    'redshift_unc': actdr6['redshiftErr'],
    'bcg_specz': -1,
    'richness': -1,
    'richness_unc': -1,
    'M200m': actdr6['M200m'] * 1e14,
    'M500c': actdr6['M500c'] * 1e14,
    'catalog_source': 'ACT DR6',
    'catalog_id': actdr6['name'],
    })

In [ ]:
planck_clusters_df = pd.DataFrame({
    'RA': planck['RA'],
    'DEC': planck['DEC'],
    'redshift': planck['redshift'],
    'redshift_type': -1,
    'redshift_unc': -1,
    'bcg_specz': -1,
    'richness': -1,
    'richness_unc': -1,
    'M200m': -1,
    'M500c': planck['msz'] * 1e14,
    'catalog_source': 'Planck',
    'catalog_id': planck['index'],
    })

In [ ]:
des_clusters_df = pd.DataFrame({
    'RA': f_des['catalog']['cluster']['ra'][:].byteswap().view(f_des['catalog']['cluster']['ra'].dtype.newbyteorder()),
    'DEC': f_des['catalog']['cluster']['dec'][:].byteswap().view(f_des['catalog']['cluster']['dec'].dtype.newbyteorder()),
    'redshift': f_des['catalog']['cluster']['z_lambda'][:].byteswap().view(f_des['catalog']['cluster']['z_lambda'].dtype.newbyteorder()),
    'redshift_type': -1,
    'redshift_unc': f_des['catalog']['cluster']['z_lambda_e'][:].byteswap().view(f_des['catalog']['cluster']['z_lambda_e'].dtype.newbyteorder()),
    'bcg_specz': f_des['catalog']['cluster']['bcg_spec_z'][:].byteswap().view(f_des['catalog']['cluster']['bcg_spec_z'].dtype.newbyteorder()),
    'richness': f_des['catalog']['cluster']['lambda_chisq'][:].byteswap().view(f_des['catalog']['cluster']['lambda_chisq'].dtype.newbyteorder()),
    'richness_unc': f_des['catalog']['cluster']['lambda_chisq_e'][:].byteswap().view(f_des['catalog']['cluster']['lambda_chisq_e'].dtype.newbyteorder()),
    'M200m': f_des['catalog']['cluster']['m200m'][:].byteswap().view(f_des['catalog']['cluster']['m200m'].dtype.newbyteorder()),
    'M500c': -1,
    'catalog_source': 'DES Y3 RedMaPPer',
    'catalog_id': f_des['catalog']['cluster']['mem_match_id'][:].byteswap().view(f_des['catalog']['cluster']['mem_match_id'].dtype.newbyteorder()),
    })

In [ ]:
camira_deep_df = pd.DataFrame({
    'RA': camira_deep['RA'],
    'DEC': camira_deep['DEC'],
    'redshift': camira_deep['Z'],
    'redshift_type': -1,
    'redshift_unc': -1,
    'bcg_specz': camira_deep['Z_BCG'],
    'richness': camira_deep['RICHNESS'],
    'richness_unc': -1,
    'M200m': -1,
    'M500c': -1,
    'catalog_source': 'HSC Deep',
    'catalog_id': camira_deep['ID'],
    })

camira_wide_df = pd.DataFrame({
    'RA': camira_wide['RA'],
    'DEC': camira_wide['DEC'],
    'redshift': camira_wide['Z'],
    'redshift_type': -1,
    'redshift_unc': -1,
    'bcg_specz': camira_wide['Z_BCG'],
    'richness': camira_wide['RICHNESS'],
    'richness_unc': -1,
    'M200m': -1,
    'M500c': -1,
    'catalog_source': 'HSC Wide',
    'catalog_id': camira_wide['ID'],
    })

In [ ]:
full_clusters_catalog = pd.concat([des_clusters_df, spt_clusters_df, mcxc_clusters_df, act_clusters_df, planck_clusters_df, camira_deep_df, camira_wide_df, erass_df], ignore_index=True)

# Assign clusters to DP2 tracts

In [ ]:
def radec_to_xyz(ra, dec):
    # Convert RA/Dec [deg] to Cartesian unit vectors
    ra = np.radians(ra)
    dec = np.radians(dec)
    cosd = np.cos(dec)
    return np.stack([cosd * np.cos(ra), cosd * np.sin(ra), np.sin(dec)], axis=-1)

def point_in_convex_spherical_polygon(point_xyz, vertices_xyz, tol=1e-12):
    # Check if point is inside a convex spherical polygon
    signs = []
    nvert = len(vertices_xyz)
    for i in range(nvert):
        v1 = vertices_xyz[i]
        v2 = vertices_xyz[(i + 1) % nvert]
        normal = np.cross(v1, v2)
        s = np.dot(normal, point_xyz)
        if abs(s) > tol:
            signs.append(np.sign(s))
    if len(signs) == 0:
        return True
    signs = np.asarray(signs)
    return np.all(signs > 0) or np.all(signs < 0)

class SkyPatchMatcher:
    def __init__(self, patches):
        """patches = [
                [[ra1, dec1], [ra2, dec2], ...],
                [[ra1, dec1], [ra2, dec2], ...],
                ...
            ]
        """
        self.patches = patches
        centers = []
        radii = []
        verts_xyz = []

        for patch in patches:
            xyz = np.array([radec_to_xyz(ra, dec) for ra, dec in patch])

            verts_xyz.append(xyz)
            center = xyz.mean(axis=0)
            center /= np.linalg.norm(center)
            centers.append(center)
            
            dots = np.clip(xyz @ center, -1.0, 1.0)
            radius = np.max(np.arccos(dots))
            radii.append(radius)

        self.centers = np.array(centers)
        self.radii = np.array(radii)
        self.verts_xyz = verts_xyz

        self.tree = cKDTree(self.centers)
        self.max_radius = np.max(self.radii)

    def find_patch(self, ra, dec):
        p = radec_to_xyz(ra, dec)
        search_radius = 2.0 * np.sin(self.max_radius / 2.0)
        candidates = self.tree.query_ball_point(p, r=search_radius)

        for idx in candidates:
            center_dot = np.clip(np.dot(self.centers[idx], p), -1.0, 1.0)
            angle = np.arccos(center_dot)
            if angle > self.radii[idx]:
                continue
            if point_in_convex_spherical_polygon(p, self.verts_xyz[idx]):
                return idx
        return -1

    def find_patches(self, ra, dec):
        out = np.full(len(ra), -1, dtype=int)
        for i, (r, d) in enumerate(zip(ra, dec)):
            out[i] = self.find_patch(r, d)
        return out

In [ ]:
# This is a dictionary of DP2 tracts, taken from TQ and Shrihan's DP2 map viewer.
with open("dp2_dataraw_jun16.txt", "r") as data:
    dp2_tracts = ast.literal_eval(data.read())
patches = [tract['verts'] for tract in dp2_tracts]

In [ ]:
matcher = SkyPatchMatcher(patches)

In [ ]:
tractlist = []
for i in tqdm.trange(len(full_clusters_catalog)):
    cluster = full_clusters_catalog.iloc[i]
    ra = cluster['RA']
    dec = cluster['DEC']
    tract_idx = matcher.find_patch(ra, dec)
    if tract_idx >= 0:
        tractlist.append(dp2_tracts[tract_idx]['tract'])
    else:
        tractlist.append(-1)
full_clusters_catalog['dp2_tract'] = tractlist

In [ ]:
mask = full_clusters_catalog['dp2_tract'] > 0
plt.scatter(full_clusters_catalog['RA'], full_clusters_catalog['DEC'], s=1)
plt.scatter(full_clusters_catalog['RA'][mask], full_clusters_catalog['DEC'][mask])
plt.gca().invert_xaxis()

In [ ]:
# Uncomment this cell to save the full clusters catalog with DP2 tract information to a CSV file:

# full_clusters_catalog.to_csv('dp2_cluster_overlap_jul23.csv', index=False)